In [37]:
!mkdir oxford-iiit-pet

!wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz

mkdir: cannot create directory ‘oxford-iiit-pet’: File exists
--2026-03-30 13:03:40--  https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://thor.robots.ox.ac.uk/pets/images.tar.gz [following]
--2026-03-30 13:03:41--  https://thor.robots.ox.ac.uk/pets/images.tar.gz
Resolving thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)... 129.67.95.98
Connecting to thor.robots.ox.ac.uk (thor.robots.ox.ac.uk)|129.67.95.98|:443... connected.
^C
--2026-03-30 13:03:41--  https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://tho

In [ ]:
!tar -xvzf images.tar.gz
!tar -xvzf annotations.tar.gz

In [ ]:
!mv images oxford-iiit-pet/
!mv annotations oxford-iiit-pet/

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split

# from model import *
# from dataloader import *

import torch.nn.functional as F
import pathlib

from tqdm import tqdm

In [2]:
EPOCHS = 50
BATCH_SIZE = 50

train_ratio = 0.8

In [3]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, dataset

import os
import numpy as np

import pathlib

from PIL import Image
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import torch.nn.functional as F


def get_class_map(root):
    class_map = {}
    with open(root / "annotations" / "list.txt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            name, class_id, species, breed_id = line.strip().split()
            class_id = int(class_id) - 1

            # Store mapping (only once per class_id)
            breed_name = "_".join(name.split("_")[:-1])

            if class_id not in class_map:
                class_map[class_id] = breed_name
    return class_map

class OxfordIIITPetDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        self.root = pathlib.Path(root_dir)
        self.images_dir = self.root / "images"
        self.xml_dir = self.root / "annotations" / "xmls"
        self.trimap_dir = self.root / "annotations" / "trimaps"
        self.transforms = transforms

        self.class_map = {}

        # Read list.txt
        self.samples = []
        with open(self.root / "annotations" / "list.txt") as f:
            for line in f:
                if line.startswith("#"):
                    continue
                name, class_id, species, breed_id = line.strip().split()
                class_id = int(class_id) - 1

                # Store mapping (only once per class_id)
                breed_name = "_".join(name.split("_")[:-1])

                if class_id not in self.class_map:
                    self.class_map[class_id] = breed_name

                img_path = self.images_dir / f"{name}.jpg"
                xml_path = self.xml_dir / f"{name}.xml"
                trimap_path = self.trimap_dir / f"{name}.png"

                if img_path.exists() and xml_path.exists() and trimap_path.exists():
                        self.samples.append({
                            "name": name,
                            "class_id": class_id,
                            "species": int(species),
                            "breed_id": int(breed_id)
                        })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        name = sample["name"]

        # Paths
        img_path = self.images_dir / f"{name}.jpg"
        xml_path = self.xml_dir / f"{name}.xml"
        trimap_path = self.trimap_dir / f"{name}.png"

        # Load image
        image = Image.open(img_path).convert("RGB")
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1)/255.0
        image = F.interpolate(image.unsqueeze(0), [224, 224], mode='bilinear')
        mean = torch.tensor([0.485, 0.456, 0.406], device=image.device).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=image.device).view(3, 1, 1)
        image = (image - mean) / std

        # Load mask
        mask = Image.open(trimap_path)
        mask = torch.torch.from_numpy(np.array(mask))/255.0
        mask = F.interpolate(mask.unsqueeze(0).unsqueeze(0), [224,224], mode='bilinear', align_corners=False)[0]

        # Parse XML
        tree = ET.parse(xml_path)
        root = tree.getroot()

        bbox = root.find("object").find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        orig_w, orig_h = Image.open(img_path).size
        H, W = 224, 224

        xc = (xmin + xmax) / 2 / orig_w
        yc = (ymin + ymax) / 2 / orig_h
        w  = (xmax - xmin) / W
        h  = (ymax - ymin) / H

        bbox = torch.tensor([xc, yc, w, h], dtype=torch.float32)

        scale_x = 224 / orig_w
        scale_y = 224 / orig_h

        xmin_224 = xmin * scale_x
        xmax_224 = xmax * scale_x
        ymin_224 = ymin * scale_y
        ymax_224 = ymax * scale_y

        xc_224 = (xmin_224 + xmax_224) / 2
        yc_224 = (ymin_224 + ymax_224) / 2
        w_224  = (xmax_224 - xmin_224)
        h_224  = (ymax_224 - ymin_224)

        bbox_224 = torch.tensor([xc_224, yc_224, w_224, h_224], dtype=torch.float32)

        if self.transforms:
            image = self.transforms(image)

        return {
            "image": image[0],
            "name": name,
            "bbox": bbox,
            "bbox_224": bbox_224,
            "mask": mask[0],
            "species": sample["species"],   # 1=cat, 2=dog
            "breed": sample["breed_id"],
            "class_id": sample["class_id"],
            "breed_name": self.class_map[sample["class_id"]]
        }

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

 
class CustomDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p
 
    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        mask = (torch.rand(x.shape, device=x.device) > self.p).to(x.dtype)
        mask = mask / ((1 - self.p) + 1e-8)
        return x * mask
 
class VGG11Classifier(nn.Module):
    def __init__(self, num_classes: int = 37, in_channels: int = 3, dropout_p: float = 0.5):
        
        super(VGG11Classifier, self).__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.dropout_p = dropout_p
 
        self.conv_layers = nn.Sequential(
            nn.Conv2d(self.in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
 
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
 
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
 
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
 
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
 
        self.linear_layers = nn.Sequential(
            nn.Linear(in_features=512 * 7 * 7, out_features=4096),
            nn.ReLU(),
            CustomDropout(p=dropout_p),
            nn.Linear(in_features=4096, out_features=4096),
            nn.ReLU(),
            CustomDropout(p=dropout_p),
            nn.Linear(in_features=4096, out_features=self.num_classes)
        )
 
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.linear_layers(x)
        return x
 

In [4]:
dataset = OxfordIIITPetDataset(root_dir = "oxford-iiit-pet")
len(dataset)

3671

In [5]:
dataset = OxfordIIITPetDataset(root_dir = "oxford-iiit-pet")

train_ds, test_ds = random_split(dataset, [int(train_ratio * len(dataset)), len(dataset)-int(train_ratio * len(dataset))])

train_dl = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size = BATCH_SIZE, shuffle=False)

In [8]:
model = VGG11Classifier(in_channels = 3, num_classes = 37, dropout_p = 0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
loss_fn = nn.CrossEntropyLoss()

train_loss_tracker = []
test_loss_tracker = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

 

VGG11Classifier(
  (conv_layers): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0,

In [43]:
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
 
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        images = batch['image'].to(device)
        class_ids = batch['class_id'].to(device)
 
        logits = model(images)
        loss = loss_fn(logits, class_ids)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        train_loss += loss.item()
 
    train_loss /= len(train_dl)
    train_loss_tracker.append(train_loss)
    scheduler.step()
 
    # Test
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
 
    with torch.no_grad():
        for batch in tqdm(test_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [test] "):
            images = batch['image'].to(device)
            class_ids = batch['class_id'].to(device)
 
            logits = model(images)
            loss = loss_fn(logits, class_ids)
            test_loss += loss.item()
 
            preds = logits.argmax(dim=1)
            correct += (preds == class_ids).sum().item()
            total += class_ids.size(0)
 
    test_loss /= len(test_dl)
    test_loss_tracker.append(test_loss)
    test_acc = correct / total * 100
 
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

Epoch 1/50 [test] : 100%|██████████| 15/15 [00:06<00:00,  2.24it/s]


Epoch 1/50 | Train Loss: 3.8539 | Test Loss: 3.5989 | Test Acc: 3.13% | LR: 0.000100


Epoch 2/50 [test] : 100%|██████████| 15/15 [00:06<00:00,  2.32it/s]


Epoch 2/50 | Train Loss: 3.5945 | Test Loss: 3.4990 | Test Acc: 7.07% | LR: 0.000100


Epoch 3/50 [test] : 100%|██████████| 15/15 [00:06<00:00,  2.30it/s]


...

Epoch 48/50 | Train Loss: 0.3816 | Test Loss: 2.3903 | Test Acc: 39.05% | LR: 0.000000


Epoch 49/50 [test] : 100%|██████████| 15/15 [00:06<00:00,  2.35it/s]


Epoch 49/50 | Train Loss: 0.3695 | Test Loss: 2.3860 | Test Acc: 38.64% | LR: 0.000000


Epoch 50/50 [test] : 100%|██████████| 15/15 [00:06<00:00,  2.27it/s]

Epoch 50/50 | Train Loss: 0.3615 | Test Loss: 2.3923 | Test Acc: 38.91% | LR: 0.000000


In [9]:
model = VGG11Classifier(num_classes=37, in_channels=3)
checkpoint = torch.load("vgg11_pet_model_30_epochs.pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

VGG11Classifier(
  (conv_layers): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0,

In [10]:
mappings = get_class_map(pathlib.Path("oxford-iiit-pet"))

In [11]:
correct = 0
total = 0

for sample in test_dl:
    num = sample['image'].shape[0]
    total += num
    for idx in range(num):
        sample_image = sample['image'][idx]
        sample_class_id = sample['class_id'][idx]

        sample_pred_logits = model(sample_image.unsqueeze(0))

        sample_preds = torch.softmax(sample_pred_logits, dim = 1)
        sample_class_id_pred = torch.argmax(sample_preds)

        print(f"actual   : {sample_class_id} | {mappings[sample_class_id.item()]}")
        print(f"predicted: {sample_class_id_pred} | {mappings[sample_class_id_pred.item()]}\n")

        if sample_class_id == sample_class_id_pred:
            correct += 1

actual   : 19 | leonberger
predicted: 19 | leonberger

actual   : 11 | Egyptian_Mau
predicted: 16 | havanese

actual   : 23 | Persian
predicted: 23 | Persian

actual   : 25 | pug
predicted: 25 | pug

actual   : 20 | Maine_Coon
predicted: 0 | Abyssinian

actual   : 9 | British_Shorthair
predicted: 9 | British_Shorthair

actual   : 17 | japanese_chin
predicted: 17 | japanese_chin

actual   : 17 | japanese_chin
predicted: 17 | japanese_chin

actual   : 20 | Maine_Coon
predicted: 0 | Abyssinian

actual   : 20 | Maine_Coon
predicted: 20 | Maine_Coon

actual   : 7 | Bombay
predicted: 7 | Bombay

actual   : 17 | japanese_chin
predicted: 28 | saint_bernard

actual   : 4 | beagle
predicted: 4 | beagle

actual   : 20 | Maine_Coon
predicted: 20 | Maine_Coon

actual   : 22 | newfoundland
predicted: 22 | newfoundland

actual   : 22 | newfoundland
predicted: 22 | newfoundland

actual   : 30 | scottish_terrier
predicted: 7 | Bombay

actual   : 33 | Sphynx
predicted: 33 | Sphynx

actual   : 28 | saint

KeyboardInterrupt: 

In [131]:
(correct/total)*100

72.6530612244898

In [132]:
import torch
from torch import nn

class IoULoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        # pred, target: [B, 4] -> (xc, yc, w, h)

        # Convert to corners
        pred_x1 = pred[:, 0] - pred[:, 2] / 2
        pred_y1 = pred[:, 1] - pred[:, 3] / 2
        pred_x2 = pred[:, 0] + pred[:, 2] / 2
        pred_y2 = pred[:, 1] + pred[:, 3] / 2

        target_x1 = target[:, 0] - target[:, 2] / 2
        target_y1 = target[:, 1] - target[:, 3] / 2
        target_x2 = target[:, 0] + target[:, 2] / 2
        target_y2 = target[:, 1] + target[:, 3] / 2

        # Intersection
        x1 = torch.max(pred_x1, target_x1)
        y1 = torch.max(pred_y1, target_y1)
        x2 = torch.min(pred_x2, target_x2)
        y2 = torch.min(pred_y2, target_y2)

        inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

        # Areas
        pred_area = (pred_x2 - pred_x1) * (pred_y2 - pred_y1)
        target_area = (target_x2 - target_x1) * (target_y2 - target_y1)

        union = pred_area + target_area - inter + 1e-6

        iou = inter / union

        return 1 - iou.mean()

In [134]:
class VGG11_Localizer(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 1024),
            nn.ReLU(),
            nn.Linear(1024, 4),
            nn.Sigmoid()  # ensures output in [0,1]
        )

    def forward(self, x):
        x = self.backbone(x)
        bbox = self.regressor(x)
        return bbox

In [139]:
model_localizer = VGG11_Localizer(model.conv_layers)
model_localizer = model_localizer.to(device)
model_localizer(torch.rand(1, 3, 224, 224))

tensor([[0.5562, 0.4617, 0.4063, 0.4479]], grad_fn=<SigmoidBackward0>)

In [ ]:
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
 
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        images = batch['image'].to(device)
        class_ids = batch['class_id'].to(device)
        bbox_224 = batch['bbox_224'].to(device)
 
        pred_bbox = model(images)
        loss = loss_fn(pred_bbox, bbox_224)
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        train_loss += loss.item()
 
    train_loss /= len(train_dl)
    train_loss_tracker.append(train_loss)
    scheduler.step()
 
    # Test
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
 
    with torch.no_grad():
        for batch in tqdm(test_dl, desc=f"Epoch {epoch+1}/{EPOCHS} [test] "):
            images = batch['image'].to(device)
            class_ids = batch['class_id'].to(device)
            bbox_224 = batch['bbox_224'].to(device)
 
            pred_bbox = model(images)
            loss = loss_fn(pred_bbox, bbox_224)

            test_loss += loss.item()
 
            preds = logits.argmax(dim=1)
            correct += (preds == class_ids).sum().item()
            total += class_ids.size(0)
 
    test_loss /= len(test_dl)
    test_loss_tracker.append(test_loss)
    test_acc = correct / total * 100
 
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}% | LR: {scheduler.get_last_lr()[0]:.6f}")